# define prompt

In [ ]:
SYSTEM_PROMPT = """
/no_think
Map greek text words from the provided greek text to their respective latvian text translations. 
Trust the provided locus marker, do not seek bible verse.
Preserve greek word order. Output only valid full JSON with all the words from greek text.
Output format:
{
  "locus": "string",
  "greek_words": [
    {
      "index": integer,
      "greek": "string",
      "latvian": ["string", "..."]
    }
  ],
  "leftover_latvian": ["string", "..."]
}

If Latvian words remain unmapped, include them in leftover_latvian.
"""

SYSTEM_PROMPT = """
You are a precise linguistic alignment engine. For each greek word find zero or more corresponding latvian words
Return only valid JSON. No explanations. No comments. Do not seek extra translations, but pick only from the words provided.

INPUT FORMAT:
First line: Latvian verse.
After newline: Greek verse.
That is the same verse in both languages.
/no_think
"""

SYSTEM_PROMPT = """
Map greek text words from the provided greek text to their respective latvian text translations. 
split by whitespace.
If you encounter pipe symbol | that is to be treated as single unit it is two variations of same text, keep it as one unit in the result too. Greek words are in correct order. for each of greek words find zero or more coressponding latvian words.
Preserve greek word order. Output only valid full JSON with all the words from greek text.
Output format:
{
  "greek_words": [
    {
      "index": integer,
      "greek": "string",
      "latvian": ["string", "..."]
    }
  ],
  "leftover_latvian": ["string", "..."]
}

If Latvian words remain unmapped, include them in leftover_latvian.
"""

In [182]:
from openai import OpenAI
import json

client = OpenAI(
    base_url="http://localhost:11435/v1",
    api_key="ollama"
)

def get_alignment(listtext):
    #user_input = latvian_text.strip() + "\n" + greek_text.strip()

    msgs = [
#        prompt= SYSTEM_PROMPT+"\n"+text.strip()#[
            {"role": "system", "content": SYSTEM_PROMPT}
        ]
    for i in listtext:
        #msgs.append({"role": "user", "content": 
                     #f'locus: ```\n{i["locus"]}\n```\ngreek text: ```\n{i["gk"]}\n```\nlatvian text: ```\n{i["lv"]}\n```'})
        msgs.append({"role": "user", "content": 
                     f'{i["lv"]}\n{i["gk"]}\n'})

    #print("\n".join(json.dumps(i) for i in msgs))
    response = client.chat.completions.create(
#        model="qwen2.5:72b",
#        model="llama3:70b",
        model="qwen3:30b-a3b",
        temperature=0.1,
#        max_tokens=500,   # IMPORTANT
        messages = msgs,
        top_p = 0.1,
#        stop=["\n", "```"]
    )

    msg = response.choices[0].message.content
    #msg = response.choices[0].text
    try:
        return json.loads(msg)
    except:
        print(f"message of length: {len(msg)}\n {msg}")
        return json.loads(msg)

# get local data

## clean data text

In [61]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
lv65_df = pd.read_csv("latvian_full65.csv")
import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFC', text).translate(d)
print(strongs_df["form"].apply(raccs_common).head())

0       Παῦλος
1       κλητὸς
2    ἀπόστολος
3      Χριστοῦ
4        Ἰησοῦ
Name: form, dtype: str


## morphology definition

In [62]:
POS_MAP = {
    "V": "Verb",
    "N": "Noun",
    "Adv": "Adverb",
    "Adj": "Adjective",
    "Art": "Article",
    "DPro": "Demonstrative Pronoun",
    "IPro": "Interrogative / Indefinite Pronoun",
    "PPro": "Personal / Possessive Pronoun",
    "RecPro": "Reciprocal Pronoun",
    "RelPro": "Relative Pronoun",
    "RefPro": "Reflexive Pronoun",
    "Prep": "Preposition",
    "Conj": "Conjunction",
    "I": "Interjection",
    "Prtcl": "Particle",
    "Heb": "Hebrew Word",
    "Aram": "Aramaic Word"
}

PERSON_MAP = {"1": "1st Person", "2": "2nd Person", "3": "3rd Person"}
TENSE_MAP = {"P": "Present", "I": "Imperfect", "F": "Future",
             "A": "Aorist", "R": "Perfect", "L": "Pluperfect"}
MOOD_MAP = {"I": "Indicative", "M": "Imperative",
            "S": "Subjunctive", "O": "Optative",
            "N": "Infinitive", "P": "Participle"}
VOICE_MAP = {"A": "Active", "M": "Middle", "P": "Passive", "M/P": "Middle or Passive"}
CASE_MAP = {"N": "Nominative", "V": "Vocative", "A": "Accusative",
            "G": "Genitive", "D": "Dative"}
NUMBER_MAP = {"S": "Singular", "P": "Plural"}
GENDER_MAP = {"M": "Masculine", "F": "Feminine", "N": "Neuter"}
COMPARISON_MAP = {"C": "Comparative", "S": "Superlative"}

In [63]:
def parse_morph_code(code):

    if not isinstance(code, str) or "-" not in code:
        return {}

    parts = code.split("-")
    pos = parts[0]

    result = {"part_of_speech": POS_MAP.get(pos, pos)}

    # Verb pattern: V-TVM-PN
    if pos == "V" and len(parts) >= 3:
        tvm = parts[1]
        pn = parts[2]

        if len(tvm) >= 3:
            result["tense"] = TENSE_MAP.get(tvm[0])
            result["voice"] = VOICE_MAP.get(tvm[1])
            result["mood"] = MOOD_MAP.get(tvm[2])

        if len(pn) == 2:
            result["person"] = PERSON_MAP.get(pn[0])
            result["number"] = NUMBER_MAP.get(pn[1])

        return result

    # Noun/Adj/Pronoun pattern: POS-CNG
    if len(parts) >= 2:
        cng = parts[1]

        if len(cng) >= 1:
            result["case"] = CASE_MAP.get(cng[0])
        if len(cng) >= 2:
            result["number"] = NUMBER_MAP.get(cng[1])
        if len(cng) >= 3:
            result["gender"] = GENDER_MAP.get(cng[2])
        if len(cng) >= 4:
            result["comparison"] = COMPARISON_MAP.get(cng[3])

    return result

In [64]:
def build_token_object(row, latvian_words):

    morph_code = row["en_title"]   # your morphology column

    return {
        "greek_form": row["form"],
        "latvian_words": latvian_words,
        "strong_number": row["strong_num"],
        "strong_title": row["strong_title"],
        "morph_code": morph_code,
        "morph_parsed": parse_morph_code(morph_code)
    }

## now get data

In [205]:
import pandas as pd
import time
from pathlib import Path

def prepare_grouped(strongs_df, lv_df):
    strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
    lv_g = lv_df.groupby(["book", "chapter", "verse"])
    return strongs_g, lv_g

def build_verse_object(book, chapter, verse, strongs_g, lv_g):

    key = (book, chapter, verse)

    if key not in strongs_g.groups:
        return None

    if key not in lv_g.groups:
        return None

    strongs_verse = strongs_g.get_group(key)

    latvian_text = lv_g.get_group(key).iloc[0]["text"]
    print(strongs_verse.sort_values("word"))

    greek_text = " ".join([t["form"] for t in strongs_verse.sort_values("word")])

    return {
        "book": book,
        "chapter": chapter,
        "verse": verse,
        "greek_text": greek_text,
        "latvian_text": latvian_text,
    }

def process_book_full(book, strongs_g, lv_g, exclude_list=[], stopatchap=-1):
    results = []
    total_start = time.perf_counter()   # ⏳ start total timer
    # Drive iteration from Latvian (verse-level dataset)
    keyss = sorted(lv_g.groups.keys())
    for i, key in enumerate(keyss):
        b, chapter, verse = key
        if b != book:
            continue
        #print(key)
        #print(exclude_list)
        #print(stopatchap)
        #break

        
        if key in exclude_list:
            continue

        #current chunk limit
        if stopatchap>0 and chapter >= stopatchap:
            break

        base_path = Path("bible") / str(b) / str(chapter)
        base_path.mkdir(parents=True, exist_ok=True)
        
        latvian_text = lv_g.get_group(key).iloc[0]["text"]
        greek_text = " ".join([raccs_common(t["form"]) for _, t in strongs_g.get_group(key).sort_values("word").iterrows()])
        request_start = time.perf_counter()   # ⏱ start request timer
        print(f"\n{greek_text}\n{latvian_text}\n")
        result = get_alignment([{
            "locus": f"{b}_{chapter}_{verse}",
#            "gk": "{"+f"{" ,".join(f"\{{str(i)}: {w}\}" for i, w in enumerate(greek_text.split()))}"+"}",
#            "lv": "{"+f"{" ,".join(f"\{{str(i)}: {w}\}" for i, w in enumerate(latvian_text.split()))}"+"}"
            "gk": greek_text,
            "lv": latvian_text
        }])
        print(f"{len(strongs_g.get_group(key))} / {len(result["greek_words"])}")

        
        request_elapsed = time.perf_counter() - request_start
        total_elapsed = time.perf_counter() - total_start
        print(f"{b}_{chapter}_{verse} [req: {request_elapsed:.2f}s | total: {total_elapsed:.2f}s]")
        print(f"{result}\n")
        
        results.append(result)
        with open(f"{Path("bible") / str(b) / str(chapter) / f'{b}_{chapter}_{verse}.txt'}", "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

    return results

#cache index
strongs_g, lv_g = prepare_grouped(strongs_df, lv65_df)
#get book

#print(book_data[0])   

## john

In [ ]:
ex_list = [("john", 1, i) for i in range (1, 51+1)] # if i != 42]
ex_list.extend([("john", 2, i) for i in range (1, 25+1)])
ex_list.extend([("john", 3, i) for i in range (1, 36+1)])
ex_list.extend([("john", 4, i) for i in range (1, 54+1)])
ex_list.extend([("john", 5, i) for i in range (1, 47+1)])
ex_list.extend([("john", 6, i) for i in range (1, 71+1)])
ex_list.extend([("john", 7, i) for i in range (1, 53+1)])
ex_list.extend([("john", 8, i) for i in range (1, 59+1)])
ex_list.extend([("john", 9, i) for i in range (1, 41+1)])
ex_list.extend([("john", 10, i) for i in range (1, 42+1)])
ex_list.extend([("john", 11, i) for i in range (1, 57+1)])
ex_list.extend([("john", 12, i) for i in range (1, 50+1)])
ex_list.extend([("john", 13, i) for i in range (1, 38+1)])
ex_list.extend([("john", 14, i) for i in range (1, 31+1)])
ex_list.extend([("john", 15, i) for i in range (1, 27+1)])
ex_list.extend([("john", 16, i) for i in range (1, 33+1)])
ex_list.extend([("john", 17, i) for i in range (1, 26+1)])
ex_list.extend([("john", 18, i) for i in range (1, 40+1)])
ex_list.extend([("john", 19, i) for i in range (1, 42+1)])
ex_list.extend([("john", 20, i) for i in range (1, 31+1)])
ex_list.extend([("john", 21, i) for i in range (1, 25+1)])
#data = process_book_full(
    "john",
    strongs_g,
    lv_g,
    ex_list,
    -1
)


## mark

In [ ]:
ex_list = [("mark", 1, i) for i in range (1, 45+1)] # if i != 42]
ex_list.extend([("mark", 2, i) for i in range (1, 28+1)])
ex_list.extend([("mark", 3, i) for i in range (1, 35+1)])
ex_list.extend([("mark", 4, i) for i in range (1, 41+1)])
ex_list.extend([("mark", 5, i) for i in range (1, 43+1)])
ex_list.extend([("mark", 6, i) for i in range (1, 56+1)])
ex_list.extend([("mark", 7, i) for i in range (1, 37+1)])
ex_list.extend([("mark", 8, i) for i in range (1, 38+1)])
ex_list.extend([("mark", 9, i) for i in range (1, 50+1)])
ex_list.extend([("mark", 10, i) for i in range (1, 52+1)])
ex_list.extend([("mark", 11, i) for i in range (1, 33+1)])
ex_list.extend([("mark", 12, i) for i in range (1, 44+1)])
ex_list.extend([("mark", 13, i) for i in range (1, 37+1)])
ex_list.extend([("mark", 14, i) for i in range (1, 14+1)])
data = process_book_full(
    "mark",
    strongs_g,
    lv_g,
    ex_list,
    -1
)

d1 = process_book_full(
    "acts",
    strongs_g,
    lv_g,
    ex_list,
    -1
)


καὶ αὐτὸς ὑμῖν δείξει ἀνάγαιον μέγα ἐστρωμένον ἕτοιμον καὶ ἐκεῖ ἑτοιμάσατε ἡμῖν
Un viņš jums rādīs lielu segām izklātu sagatavotu istabu Sataisiet tur mums visu

